In [ ]:
!pip install -q -U google-genai langchain langchain-community langchain-text-splitters langchain-huggingface sentence-transformers faiss-cpu

In [ ]:
import os
import json
import time
from datetime import datetime
from google import genai
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_FOLDER = "/content/drive/MyDrive/Graduation Project/قانون المرافعات/Cleaned"
DB_FOLDER = "/content/drive/MyDrive/Graduation Project/Vector DataBase/قانون المرافعات"

In [ ]:
client = genai.Client(api_key=userdata.get('GOOGLE_API_1'))

In [ ]:
# Cell 5: Define metadata extraction function using Gemini
def extract_metadata(file_path, filename, first_n_lines=50):
    """
    Extracts metadata using Gemini API by analyzing filename and first 50 lines.
    Returns dict, handles nulls.
    """
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = [f.readline() for _ in range(first_n_lines)]
        content_text = "".join(lines)
    except Exception as e:
        print(f"❌ File Error for {filename}: {e}")
        return None

    prompt = f"""
    You are a legal data architect. I am providing you with two pieces of information:
    1. A Filename: "{filename}"
    2. The first {first_n_lines} lines of the document content.

    INSTRUCTIONS:
    - Analyze the filename: If it contains useful words (e.g., book title, author), use it to help identify the book.
    - Analyze the text content: Cross-reference it with the filename.
    - Correct OCR/Typos: Fix any spelling errors in Arabic.

    Return ONLY a JSON object:
    {{
        "filename": "{filename}",
        "book_title": "string or null",
        "author": "string or null",
        "publisher": "string or null",
        "publication_year": "string or null",
        "edition": "string or null",
        "part_number": "string or null",
        "topic": "Brief summary in arabic",
        "law_area": "قانون المرافعات المدني",
        "source": "{filename}"
    }}

    TEXT CONTENT:
    {content_text}
    """

    try:
        response = client.models.generate_content(
            model="models/gemini-2.5-flash",
            contents=prompt ,
            config={"response_mime_type": "application/json"}
        )

        if response:
            print(f"✅ Success for {filename}! Response received.")
            total_tokens = response.usage_metadata.prompt_token_count + response.usage_metadata.candidates_token_count
            print(f"📊 Tokens used for {filename}: {total_tokens}")
            metadata = json.loads(response.text)
            # Handle nulls
            for key, value in metadata.items():
                if value is None:
                    metadata[key] = "غير متوفر"
            return metadata
    except Exception as e:
        error_str = str(e)
        if "429" in error_str or "quota" in error_str.lower():
            print(f"🛑 QUOTA EXCEEDED for {filename}. Stopping processing.")
            raise StopIteration("Quota exceeded - stopping all processing.")
        else:
            print(f"⚠️ Error for {filename}: {e}")
        # Fallback
        return {
            "filename": filename,
            "book_title": filename,
            "author": "غير متوفر",
            "publisher": "غير متوفر",
            "publication_year": "غير متوفر",
            "edition": "غير متوفر",
            "part_number": "غير متوفر",
            "topic": "غير متوفر",
            "law_area": "مدني",
            "source": "file",
            "extraction_date": datetime.now().strftime("%Y-%m-%d")
        }

In [ ]:
def print_metadata_summary(metadata):
    """Prints a nice summary of metadata."""
    print("\n" + "="*60)
    print("Metadata Summary")
    print("="*60)
    print(f"📖 العنوان: {metadata['book_title']}")
    print("-"*60)
    print(f"✍️  المؤلف: {metadata['author']}")
    print("-"*60)
    print(f"📅 سنة النشر: {metadata['publication_year']}")
    print("-"*60)
    print(f"🏢 الناشر: {metadata['publisher']}")
    print("-"*60)
    print(f"📑 الطبعة: {metadata['edition']}")
    print("-"*60)
    print(f"📚 رقم الجزء: {metadata['part_number']}")
    print("-"*60)
    print(f"🎯 الموضوع: {metadata['topic']}")
    print("-"*60)
    print(f"⚖️  نوع القانون: {metadata['law_area']}")
    print("-"*60)
    print(f"📁 اسم الفايل: {metadata['filename']}")
    print("="*60 + "\n")

In [ ]:
# Embeddings
embeddings = HuggingFaceEmbeddings(
              model_name="medmediani/Arabic-KW-Mdel",
              encode_kwargs={'normalize_embeddings': True}
          )

In [ ]:
# Cell 7: Main processing loop for folder (batch sequential due to API limits)
def process_folder(input_folder, db_folder):
    """
    Processes all .txt files in input_folder.
    Skips if DB already exists.
    Stops on quota exceed.
    Adds sleep between files.
    """
    files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
    print(f"Found {len(files)} files to process.")

    for file in files:  # Sequential batch
        file_path = os.path.join(input_folder, file)
        db_name = os.path.splitext(file)[0] + "_vectordb"
        db_path = os.path.join(db_folder, db_name)

        # Skip if DB exists
        if os.path.exists(db_path):
            print(f"⏭️ Skipping {file} - Vector DB already exists at {db_path}")
            continue

        print(f"\n🔄 Processing {file}...")

        # Extract metadata with Gemini
        metadata = extract_metadata(file_path, file)
        if metadata is None:
            print(f"❌ Skipping {file} due to extraction error.")
            continue

        print_metadata_summary(metadata)

        # Load document
        try:
            loader = TextLoader(file_path, encoding="utf-8")
            documents = loader.load()
            # Add metadata to docs
            for doc in documents:
                doc.metadata = metadata.copy()
            print(f"✅ Loaded {file} successfully!")
        except Exception as e:
            print(f"❌ Error loading {file}: {e}")
            continue

        # Split into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
            separators=["\n\n", "\n", "المادة", "الباب", "الفصل", "،", "؛", "."]
        )
        chunks = text_splitter.split_documents(documents)
        print(f"✅ Created {len(chunks)} chunks for {file}")

        # Build and save Vector DB
        vector_db = FAISS.from_documents(chunks, embeddings)
        vector_db.save_local(db_path)
        print(f"✅ Saved Vector DB for {file} at {db_path}")

        # Sleep to respect rates (e.g., 10s between files)
        time.sleep(10)

# Run the processing
try:
    process_folder(INPUT_FOLDER, DB_FOLDER)
except StopIteration as e:
    print(f"🛑 Processing stopped: {e}")
print("🏁 All processing complete!")